## 1. Hugging Face Login

#### This imports the token from Kaggle Secrets, so make sure the secret is set correctly before running the code.

In [3]:
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


## 2. Check GPU

In [4]:
import torch

print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

CUDA available: True
GPU count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


## 3. Configuration

#### this is sample configuration, you are free to modify any parameter to train the best performing model

In [5]:
import os
import random
import json
import time
import math
import zipfile
import shutil

# Hugging Face dataset
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

# You can use any of the 4 train files or all the files for pre-training the legal model
TRAIN_FILE = "Phase_1_Pretraining/Train/train_supreme_court.json"
TEST_FILE = "Phase_1_Pretraining/Test/test.json"

# Training configuration
TRAIN_TOKENS = 200_000_000
VAL_TOKENS = 5_000_000
TRAIN_BLOCK_SIZE = 2048
EPOCHS = 1
SEED = 42
random.seed(SEED)
MODEL_CONTEXT_LENGTH = 2048
N_EMBD = 560
N_LAYER = 8
N_HEAD = 8

# Generation
TARGET_GENERATION_WORDS = 128
MAX_GENERATION_TOKENS = 384
TEMPERATURE = 0.1
TOP_P = 0.95

# Kaggle output
OUTPUT_DIR = "/kaggle/working/cs781_phase1"
MODEL_OUTPUT_DIR = os.path.join(
    OUTPUT_DIR,
    "legal-gpt2-final"
)
PREDICTION_PATH = os.path.join(
    OUTPUT_DIR,
    "prediction.json"
)
SUBMISSION_ZIP = os.path.join(
    OUTPUT_DIR,
    "submission.zip"
)
LOSS_PLOT_PATH = os.path.join(
    OUTPUT_DIR,
    "training_validation_loss.png"
)
os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)
print("Configuration loaded.")

Configuration loaded.


## 4. Load Tokenizer

In [6]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("openai-community/gpt2")
tokenizer.pad_token = tokenizer.eos_token
print("Vocabulary size:", len(tokenizer))

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Vocabulary size: 50257


In [7]:
from huggingface_hub import HfApi
for f in HfApi().list_repo_tree(HF_DATASET_REPO, repo_type="dataset",
                                path_in_repo="Phase_1_Pretraining", recursive=True):
    size = getattr(f, "size", None)
    print(f.path, f"| {size/1e9:.2f} GB" if size else "")

Phase_1_Pretraining/Test 
Phase_1_Pretraining/Train 
Phase_1_Pretraining/Test/test.json | 0.00 GB
Phase_1_Pretraining/Train/train_acts.json | 0.73 GB
Phase_1_Pretraining/Train/train_high_courts.json | 10.67 GB
Phase_1_Pretraining/Train/train_supreme_court.json | 1.32 GB
Phase_1_Pretraining/Train/train_tribunals.json | 6.48 GB


## 5. Download Training Data From Hugging Face

In [8]:
from huggingface_hub import hf_hub_download

train_file_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename=TRAIN_FILE,
    repo_type="dataset",
    token=HF_TOKEN
)

print("Training file:")
print(train_file_path)

Phase_1_Pretraining/Train/train_supreme_(…):   0%|          | 0.00/1.32G [00:00<?, ?B/s]

Training file:
/root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_1_Pretraining/Train/train_supreme_court.json


In [9]:
%%writefile a1_tokenizer.py
"""this is my scratch implementattion from assignmentt 3 of BPE tokeniser."""
import re
import json
from collections import Counter

PAT=re.compile(r" ?[A-Za-z]+| ?\d+| ?[^\sA-Za-z\d]+|\s+")
class BPETokeniser:#defining the class
    def __init__(self):
        self.merges={}
        self.vocab={i: bytes([i]) for i in range(256)}
        self.cache={}

    @property
    def vocab_size(self):#vocab size
        return len(self.vocab)

    def _count_pairs(self,words,counts):#how actuall work we start counting how many words have same pair 
        pair_counts={}
        for word,count in zip(words,counts):
            for a,b in zip(word,word[1:]):
                pair_counts[(a,b)]=pair_counts.get((a,b),0)+count
        return pair_counts
    def _merge(self,ids,pair,new_id):
        out=[]
        i=0
        while i<len(ids):
            if i+1<len(ids) and ids[i]==pair[0] and ids[i+1]==pair[1]:
                out.append(new_id)
                i+=2
            else:
                out.append(ids[i])
                i+=1
        return out
    def train_slow(self,text,vocab_size):   # Version 1: recounts every pair each round (reference)
        chunks=PAT.findall(text)
        chunk_counts=Counter(chunks)
        words=[]
        counts=[]
        for chunk,n in chunk_counts.items():
            words.append(list(chunk.encode("utf-8")))
            counts.append(n)
        for r in range(vocab_size-256):
                pair_counts=self._count_pairs(words,counts)
                if not pair_counts:
                    break
                best = min(pair_counts, key=lambda p: (-pair_counts[p], p))
                new_id=256+r
                self.merges[best]=new_id
                self.vocab[new_id]=self.vocab[best[0]] + self.vocab[best[1]]
                words=[self._merge(w,best,new_id) for w in words]
    def train(self,text,vocab_size):        # Version 2: counts once, updates only affected words
        chunks=PAT.findall(text)
        chunk_counts=Counter(chunks)
        words=[]
        counts=[]
        for chunk,n in chunk_counts.items():
            words.append(list(chunk.encode("utf-8")))
            counts.append(n)
        pair_counts={}
        where={}
        for wi,(w,c) in enumerate(zip(words,counts)):
            for a,b in zip(w,w[1:]):
                pair_counts[(a,b)]=pair_counts.get((a,b),0)+c
                where.setdefault((a,b),set()).add(wi)
        for r in range(vocab_size-256):
            if not pair_counts:
                break
            best=min(pair_counts,key=lambda p:(-pair_counts[p],p))
            new_id=256+r
            self.merges[best]=new_id
            self.vocab[new_id]=self.vocab[best[0]]+self.vocab[best[1]]
            for wi in list(where[best]):
                w,c=words[wi],counts[wi]
                for a,b in zip(w,w[1:]):
                    pair_counts[(a,b)]-=c
                    if pair_counts[(a,b)]==0:
                        del pair_counts[(a,b)]
                    where[(a,b)].discard(wi)
                w=self._merge(w,best,new_id)
                words[wi]=w
                for a,b in zip(w,w[1:]):
                    pair_counts[(a,b)]=pair_counts.get((a,b),0)+c
                    where.setdefault((a,b),set()).add(wi)
            where.pop(best,None)
    def save(self,path):
        json.dump([[a,b,i] for (a,b),i in self.merges.items()],open(path,"w"))
    def load(self,path):
        for a,b,i in json.load(open(path)):
            self.merges[(a,b)]=i
            self.vocab[i]=self.vocab[a]+self.vocab[b]
    def _encode_chunk(self,chunk):
        if chunk in self.cache:
            return self.cache[chunk]
        ids=list(chunk.encode("utf-8"))
        while True:
            candidates=[p for p in zip(ids,ids[1:]) if p in self.merges]
            if not candidates:
                break
            best=min(candidates,key=lambda p:self.merges[p])
            ids=self._merge(ids,best,self.merges[best])
        self.cache[chunk]=ids
        return ids
    def encode(self,text):
        ids=[]
        for chunk in PAT.findall(text):
            ids.extend(self._encode_chunk(chunk))
        return ids
    def decode(self,ids):
        data=b"".join(self.vocab[i] for i in ids)
        return data.decode("utf-8",errors="replace")


def train_tokenizer(train_text: str, vocab_size: int):
    tok=BPETokeniser()
    tok.train(train_text,vocab_size)
    return tok

Overwriting a1_tokenizer.py


In [10]:
import importlib, time, json, a1_tokenizer; importlib.reload(a1_tokenizer)#this is the test cell for tokenising 
with open(train_file_path, encoding="utf-8") as f:
    records = json.load(f)
sample = " ".join(r["text"] for r in records[:300])[:300_000]

t = time.time(); slow = a1_tokenizer.BPETokeniser(); slow.train_slow(sample, 600); t_slow = time.time() - t
t = time.time(); fast = a1_tokenizer.BPETokeniser(); fast.train(sample, 600);      t_fast = time.time() - t
print(f"slow {t_slow:.1f}s | fast {t_fast:.1f}s")
print("identical merges:", list(slow.merges.items()) == list(fast.merges.items()))
print([fast.vocab[i] for i in range(256, 271)])

slow 5.6s | fast 0.4s
identical merges: True
[b' t', b' th', b' the', b' a', b' o', b'in', b'er', b'on', b're', b'at', b' of', b' w', b'en', b'is', b'ed']


In [13]:
import importlib, a1_tokenizer; importlib.reload(a1_tokenizer)
with open(train_file_path, encoding="utf-8") as f:
    records = json.load(f)

TOK_SAMPLE_CHARS = 100_000_000   
LEGAL_VOCAB = 65354          

random.seed(SEED)
idx = list(range(len(records))); random.shuffle(idx)
parts, n = [], 0
for i in idx:
    parts.append(records[i]["text"]); n += len(parts[-1])
    if n >= TOK_SAMPLE_CHARS: break
tok_text = "\n".join(parts)

t0 = time.time()
tok = a1_tokenizer.BPETokeniser()
tok.train(tok_text, LEGAL_VOCAB)
tok.save(os.path.join(OUTPUT_DIR, "legal_bpe_merges.json"))
print(f"trained on {len(tok_text):,} chars ({len(parts)} docs) | vocab {tok.vocab_size} | {time.time()-t0:.0f}s")
print([tok.vocab[i] for i in range(LEGAL_VOCAB - 15, LEGAL_VOCAB)])   # the last merges learned

trained on 100,033,141 chars (4225 docs) | vocab 65354 | 1673s
[b'mugg', b'magn', b'murthy', b'nP', b'nay', b'nep', b'nond', b'nies', b'nts', b'nah', b'nya', b'narr', b'nursing', b'nidhi', b'navis']


## 6. Create Train and Validation Tokens Set

In [14]:
import numpy as np
EOS_ID = tok.vocab_size
tok.vocab[EOS_ID] = b""            # document-boundary token; decodes to empty text
VOCAB_SIZE = tok.vocab_size        # = LEGAL_VOCAB + 1  → use this for the model

random.seed(SEED)
train_chunks, val_chunks, n_train, n_val = [], [], 0, 0
t0 = time.time()
for k, data in enumerate(records):
    ids = tok.encode(data["text"]) + [EOS_ID]
    if random.random() < 0.05:
        if n_val < VAL_TOKENS:
            ids = ids[:VAL_TOKENS - n_val]; val_chunks.append(np.array(ids, dtype=np.uint16)); n_val += len(ids)
    elif n_train < TRAIN_TOKENS:
        ids = ids[:TRAIN_TOKENS - n_train]; train_chunks.append(np.array(ids, dtype=np.uint16)); n_train += len(ids)
    if n_train >= TRAIN_TOKENS and n_val >= VAL_TOKENS:
        break
    if k % 5000 == 0:
        print(f"{k:,} docs | train {n_train:,} | val {n_val:,} | {time.time()-t0:.0f}s")

train_ids = np.concatenate(train_chunks); val_ids = np.concatenate(val_chunks)
del train_chunks, val_chunks
np.save(os.path.join(OUTPUT_DIR, "train_ids_bpe.npy"), train_ids)
np.save(os.path.join(OUTPUT_DIR, "val_ids_bpe.npy"), val_ids)
print(f"DONE: train {len(train_ids):,} | val {len(val_ids):,} tokens | vocab {VOCAB_SIZE} | {time.time()-t0:.0f}s")

0 docs | train 5,000 | val 0 | 0s
5,000 docs | train 26,038,375 | val 1,507,970 | 19s
10,000 docs | train 44,508,415 | val 2,369,086 | 30s
15,000 docs | train 71,571,090 | val 3,879,093 | 47s
20,000 docs | train 87,909,687 | val 4,766,631 | 56s
25,000 docs | train 103,449,798 | val 5,000,000 | 65s
30,000 docs | train 131,076,883 | val 5,000,000 | 82s
35,000 docs | train 149,143,474 | val 5,000,000 | 92s
40,000 docs | train 172,170,720 | val 5,000,000 | 106s
45,000 docs | train 198,633,426 | val 5,000,000 | 122s
DONE: train 200,000,000 | val 5,000,000 tokens | vocab 65355 | 123s


## 7. Create Token Blocks

In [ ]:
def create_blocks(tokens, block_size):
    total_length = (len(tokens) // block_size) * block_size
    tokens = tokens[:total_length]
    return [tokens[i:i + block_size] for i in range(0, total_length, block_size)]

train_blocks = create_blocks(train_tokens, TRAIN_BLOCK_SIZE)
val_blocks = create_blocks(val_tokens, TRAIN_BLOCK_SIZE)

print("Training sequences:", len(train_blocks))
print("Validation sequences:", len(val_blocks))

## 8. Create Datasets

In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_dict({"input_ids": train_blocks})
val_dataset = Dataset.from_dict({"input_ids": val_blocks})

print(train_dataset)
print(val_dataset)

## 9. Define The Model

In [ ]:
from transformers import GPT2Config, GPT2LMHeadModel

config = GPT2Config(
    vocab_size=len(tokenizer),
    n_positions=MODEL_CONTEXT_LENGTH,
    n_ctx=MODEL_CONTEXT_LENGTH,
    n_embd=N_EMBD,
    n_layer=N_LAYER,
    n_head=N_HEAD,
    bos_token_id=tokenizer.bos_token_id,
    eos_token_id=tokenizer.eos_token_id
)

model = GPT2LMHeadModel(config)

model_size = sum(p.numel() for p in model.parameters())
print(f"Model size: {model_size / 1e6:.2f}M parameters")

In [ ]:
from transformers import DataCollatorForLanguageModeling
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

## 10. Training Arguments

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=MODEL_OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=3e-4,
    weight_decay=0.1,
    warmup_steps=500,
    lr_scheduler_type="cosine",
    fp16=torch.cuda.is_available(),
    eval_strategy="steps",
    eval_steps=1000,
    logging_steps=100,
    save_steps=1000,
    save_total_limit=2,
    seed=SEED,
    report_to="none",
    logging_dir=os.path.join(OUTPUT_DIR, "logs")
)

## 11. Training

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    processing_class=tokenizer
)

start_time = time.time()

trainer.train()

training_time = (time.time() - start_time)

tokens_per_epoch = (len(train_dataset) * TRAIN_BLOCK_SIZE)

total_tokens = (tokens_per_epoch * EPOCHS)

tokens_per_second = (total_tokens / training_time)

print(f"Training time: {training_time / 3600:.2f} hours")
print(f"Training speed: {tokens_per_second:,.0f} tokens/sec")

## 12. Perplexity

In [ ]:
eval_results = trainer.evaluate()
eval_loss = eval_results["eval_loss"]
perplexity = math.exp(eval_loss)
print(f"Validation loss: {eval_loss:.6f}")
print(f"Validation perplexity: {perplexity:.4f}")

## 13. Plot Training and Validation Loss

In [ ]:
import matplotlib.pyplot as plt

logs = trainer.state.log_history

train_steps = []
train_loss = []
eval_steps = []
eval_loss = []

for log in logs:
    if ("loss" in log and "step" in log):
        train_steps.append(log["step"])
        train_loss.append(log["loss"])
    if ("eval_loss" in log and "step" in log):
        eval_steps.append(log["step"])
        eval_loss.append(log["eval_loss"])

plt.figure(figsize=(10, 5))
plt.plot(train_steps, train_loss, label="Training Loss")
plt.plot(eval_steps, eval_loss, label="Validation Loss")
plt.xlabel("Training Steps")
plt.ylabel("Loss")
plt.title("Phase 1 Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(LOSS_PLOT_PATH, dpi=200)
plt.show()

print("Loss plot saved to:")
print(LOSS_PLOT_PATH)

## 14. Save Final Model to Kaggle Output

In [ ]:
trainer.save_model(MODEL_OUTPUT_DIR)
tokenizer.save_pretrained(MODEL_OUTPUT_DIR)

print("Final model saved to:")
print(MODEL_OUTPUT_DIR)

## 15. Push Model to Hugging Face Hub

In [ ]:
# model.push_to_hub(HF_MODEL_REPO, token=HF_TOKEN)
# tokenizer.push_to_hub(HF_MODEL_REPO, token=HF_TOKEN)

# print("Model and tokenizer pushed successfully.")

## 16. Load Test Data from HuggingFace

In [ ]:
test_file_path = hf_hub_download(
    repo_id=HF_DATASET_REPO,
    filename=TEST_FILE,
    repo_type="dataset",
    token=HF_TOKEN
)

print("Test file:", test_file_path)

## 17. Load Test Prompts

In [ ]:
with open(test_file_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

print("Number of test examples:", len(test_data))

## 18. Load Final Model for Generation

In [ ]:
from transformers import AutoTokenizer, GPT2LMHeadModel
import torch

generation_tokenizer = AutoTokenizer.from_pretrained(MODEL_OUTPUT_DIR)
generation_model = GPT2LMHeadModel.from_pretrained(MODEL_OUTPUT_DIR)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

generation_model = generation_model.to(device)
generation_model.eval()
generation_model.config.use_cache = True

print("Device:", device)
print(f"Parameters: {sum(p.numel() for p in generation_model.parameters()) / 1e6:.2f}M")

## 19. Generate 128 Words
#### (words, not tokens)

In [ ]:
def generate_batch(prompts, model, tokenizer, device, target_words=128):
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MODEL_CONTEXT_LENGTH - MAX_GENERATION_TOKENS
    )

    inputs = {key: value.to(device) for key, value in inputs.items()}
    input_lengths = inputs["attention_mask"].sum(dim=1)

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_GENERATION_TOKENS,
            do_sample=True,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            pad_token_id=tokenizer.eos_token_id,
        )

    predictions = []
    for i in range(len(prompts)):
        generated_tokens = outputs[i, input_lengths[i]:]
        generated_text = tokenizer.decode(generated_tokens, skip_special_tokens=True)
        generated_words = generated_text.split()
        generated_words = generated_words[:target_words]
        predictions.append(" ".join(generated_words))

    return predictions

## 20. Generate Predictions for the Test Set

In [ ]:
predictions = {}
GENERATION_BATCH_SIZE = 32

items = list(test_data.items())
total_examples = len(items)
print(f"Total test examples: {total_examples}")

for start in range(0, total_examples, GENERATION_BATCH_SIZE):
    batch = items[start:start + GENERATION_BATCH_SIZE]
    batch_ids = [str(example_id) for example_id, _ in batch]
    batch_prompts = [record["text"] for _, record in batch]

    batch_predictions = generate_batch(
        prompts=batch_prompts,
        model=generation_model,
        tokenizer=generation_tokenizer,
        device=device,
        target_words=128
    )

    for example_id, prediction in zip(batch_ids, batch_predictions):
        predictions[example_id] = {"text": prediction}

    completed = start + len(batch)
    print(f"Completed {completed}/{total_examples} ({100 * completed / total_examples:.1f}%)")

## 21. Save prediction.json

In [ ]:
with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
    json.dump(
        predictions,
        f,
        ensure_ascii=False,
        indent=2
    )

print("prediction.json saved:")
print(PREDICTION_PATH)

## 22. Create CodaBench Submission ZIP

In [ ]:
with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(PREDICTION_PATH, arcname="prediction.json")

print("Codabench submission created:")
print(SUBMISSION_ZIP)